# Notebook 03 — KaelvorNet

# Cross-Subject Adaptive Network for Motor-Imagery EEG (BCI-IV 2a)

---

## Before running this notebook

Same input setup as the baseline notebook:

1. **Add Input -> Notebooks -> your preprocessing notebook** (must be a completed
   "Save & Run All (Commit)" run, providing `preprocessed_dataset/A0X/A0XT.fif` / `A0XE.fif`).
2. **Add Input -> Datasets -> raw 2a dataset / true_labels bundle**, if you want the official
   T -> E split (`A0XE.mat` files). Falls back to an 80/20 split within T otherwise.

## What this notebook does

Implements KaelvorNet exactly as described in the design proposal:

1. **Euclidean Alignment (EA)** — per-subject re-centering, unsupervised
2. **Shared EEGNet-style backbone**
3. **Domain head + gradient-reversal layer (GRL)** — adversarial subject-invariance,
   training-only
4. **Subject adapter** — small per-subject BatchNorm affine parameters, few-shot fine-tunable
5. **Two evaluation regimes**:
   - Subject-dependent (train includes the target subject, same protocol as the baselines)
   - Cross-subject leave-one-subject-out (zero-shot and few-shot on a held-out subject)
6. **Ablation runner** — toggle EA / GRL / adapter on and off to reproduce the ablation table
   from the design doc

In [1]:
# Kaggle notebooks don't have mne preinstalled by default
!pip install -q mne

In [2]:
# ==========================================================
# Imports
# ==========================================================

import re
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.io as sio
import mne

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from torch.autograd import Function

from sklearn.metrics import accuracy_score, cohen_kappa_score
from sklearn.model_selection import train_test_split

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

Using device: cuda


In [3]:
# ==========================================================
# Configuration — auto-discover inputs
# ==========================================================

INPUT_ROOT = Path("/kaggle/input")
OUTPUT_PATH = Path("/kaggle/working")
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

fif_files = sorted(INPUT_ROOT.rglob("*.fif"))
mat_search_root = INPUT_ROOT

print(f"Found {len(fif_files)} preprocessed .fif file(s):")
for f in fif_files[:10]:
    print(" ", f)
if len(fif_files) > 10:
    print(f"  ... and {len(fif_files) - 10} more")

if not fif_files:

    print("\nNo .fif files found. Everything currently under /kaggle/input:\n")
    if INPUT_ROOT.exists():
        for p in sorted(INPUT_ROOT.rglob("*")):
            print(" ", p)
    else:
        print("  /kaggle/input does not exist -- no inputs attached at all.")

    raise FileNotFoundError(
        "No .fif files found under /kaggle/input. Make sure the preprocessing notebook was "
        "committed via 'Save Version -> Save & Run All (Commit)' and added here as a Notebook "
        "input (see intro cell)."
    )

CUE_CODE_TO_LABEL = {"769": 0, "770": 1, "771": 2, "772": 3}
CLASS_NAMES = {0: "Left Hand", 1: "Right Hand", 2: "Foot", 3: "Tongue"}

mi_files = [f for f in fif_files if f.stem.upper().endswith("T")]
eval_files = [f for f in fif_files if f.stem.upper().endswith("E")]

subjects_covered = sorted({f.stem[:-1] for f in mi_files})
print(f"\nSubjects with a training file: {subjects_covered}")

Found 18 preprocessed .fif file(s):
  /kaggle/input/notebooks/sababahoquesaba/eeg-preprocessing-pipeline/preprocessed_dataset/A01/A01E.fif
  /kaggle/input/notebooks/sababahoquesaba/eeg-preprocessing-pipeline/preprocessed_dataset/A01/A01T.fif
  /kaggle/input/notebooks/sababahoquesaba/eeg-preprocessing-pipeline/preprocessed_dataset/A02/A02E.fif
  /kaggle/input/notebooks/sababahoquesaba/eeg-preprocessing-pipeline/preprocessed_dataset/A02/A02T.fif
  /kaggle/input/notebooks/sababahoquesaba/eeg-preprocessing-pipeline/preprocessed_dataset/A03/A03E.fif
  /kaggle/input/notebooks/sababahoquesaba/eeg-preprocessing-pipeline/preprocessed_dataset/A03/A03T.fif
  /kaggle/input/notebooks/sababahoquesaba/eeg-preprocessing-pipeline/preprocessed_dataset/A04/A04E.fif
  /kaggle/input/notebooks/sababahoquesaba/eeg-preprocessing-pipeline/preprocessed_dataset/A04/A04T.fif
  /kaggle/input/notebooks/sababahoquesaba/eeg-preprocessing-pipeline/preprocessed_dataset/A05/A05E.fif
  /kaggle/input/notebooks/sababahoque

In [4]:
# ==========================================================
# Epoch extraction (same as the baseline notebook)
# ==========================================================

def extract_epochs(fif_file):
    """Extract labeled motor-imagery epochs from a *T file."""

    raw = mne.io.read_raw_fif(fif_file, preload=True, verbose=False)
    events, event_id = mne.events_from_annotations(raw, verbose=False)
    cue_id = {k: v for k, v in event_id.items() if k in CUE_CODE_TO_LABEL}

    if not cue_id:
        raise ValueError(f"No motor-imagery cue codes found in {fif_file.name}.")

    epochs = mne.Epochs(
        raw, events, event_id=cue_id, tmin=-0.5, tmax=4.0,
        baseline=(-0.5, 0), preload=True, reject_by_annotation=True, verbose=False
    )

    X = epochs.get_data()
    inv_cue_id = {v: k for k, v in cue_id.items()}
    y = np.array([CUE_CODE_TO_LABEL[inv_cue_id[c]] for c in epochs.events[:, 2]])

    return X, y


def extract_epochs_eval(fif_file):
    """E files mark every MI cue as 'unknown' (783); true labels attached separately."""

    raw = mne.io.read_raw_fif(fif_file, preload=True, verbose=False)
    events, event_id = mne.events_from_annotations(raw, verbose=False)
    eval_cue_id = {k: v for k, v in event_id.items() if k == "783"}

    if not eval_cue_id:
        raise ValueError(f"No 'unknown' (783) cue markers found in {fif_file.name}.")

    epochs = mne.Epochs(
        raw, events, event_id=eval_cue_id, tmin=-0.5, tmax=4.0,
        baseline=(-0.5, 0), preload=True, reject_by_annotation=True, verbose=False
    )

    return epochs.get_data()


def find_true_labels(subject_id, search_root):
    """Look for a separately-distributed true-label .mat file ('<subject_id>E.mat')."""

    candidates = list(Path(search_root).rglob(f"{subject_id}E.mat"))
    if not candidates:
        return None

    mat = sio.loadmat(candidates[0])
    key = "classlabel" if "classlabel" in mat else next(
        (k for k in mat if not k.startswith("__")), None
    )
    if key is None:
        return None

    return np.array(mat[key]).squeeze().astype(int) - 1

---
## Component 1 — Euclidean Alignment (EA)

Recenters each subject's trials by the inverse square root of that subject's mean trial
covariance matrix. Computed from unlabeled EEG alone (no class labels needed), applied
identically at train and test time for a given subject.

In [5]:
# ==========================================================
# Euclidean Alignment
# ==========================================================

from scipy.linalg import fractional_matrix_power

def compute_alignment_matrix(X):
    """
    X: (n_trials, n_channels, n_samples). Returns the (n_channels, n_channels)
    whitening matrix R^-1/2, where R is the mean trial covariance for this subject.
    """

    n_trials, n_channels, n_samples = X.shape

    covs = np.array([
        (trial @ trial.T) / n_samples for trial in X
    ])

    mean_cov = covs.mean(axis=0)

    R_inv_sqrt = fractional_matrix_power(mean_cov, -0.5).real

    return R_inv_sqrt


def apply_alignment(X, R_inv_sqrt):
    """Apply a precomputed alignment matrix to every trial."""

    return np.einsum("cd,ndt->nct", R_inv_sqrt, X)


def euclidean_align(X_ref, *X_others):
    """
    Compute the alignment matrix from X_ref (typically this subject's training trials)
    and apply it to X_ref and every array in X_others (e.g. that subject's test trials).
    Returns a list: [X_ref_aligned, *X_others_aligned].
    """

    R_inv_sqrt = compute_alignment_matrix(X_ref)

    aligned = [apply_alignment(X_ref, R_inv_sqrt)]
    for X in X_others:
        aligned.append(apply_alignment(X, R_inv_sqrt))

    return aligned

---
## Component 2 — Gradient Reversal Layer + Domain Head

The GRL is the identity function on the forward pass and negates (and scales by `lambda_`)
the gradient on the backward pass -- this is what makes the shared backbone's features
*adversarially* subject-invariant rather than merely regularized towards it.

In [6]:
# ==========================================================
# Gradient Reversal Layer
# ==========================================================

class GradientReversalFunction(Function):

    @staticmethod
    def forward(ctx, x, lambda_):
        ctx.lambda_ = lambda_
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return grad_output.neg() * ctx.lambda_, None


class GradientReversalLayer(nn.Module):

    def __init__(self, lambda_=1.0):
        super().__init__()
        self.lambda_ = lambda_

    def forward(self, x):
        return GradientReversalFunction.apply(x, self.lambda_)

---
## Component 3 — KaelvorNet Model

Shared EEGNet-style backbone -> (a) subject adapter -> MI classifier, and (b) GRL -> domain
head (subject-ID classifier, training-only). The adapter is implemented as per-subject
BatchNorm affine parameters applied to the backbone's flattened features -- cheap to fine-tune
in isolation (freeze everything else, train just this) for the few-shot cross-subject setting.

In [7]:
# ==========================================================
# KaelvorNet
# ==========================================================

class EEGNetBackbone(nn.Module):
    """Same backbone as the EEGNet baseline -- temporal conv, depthwise spatial conv,
    separable conv. Returns flattened features rather than class logits."""

    def __init__(self, n_channels=22, n_samples=1126, sfreq=250, F1=8, D=2, F2=16, dropout=0.5):
        super().__init__()

        half_sfreq = max(int(sfreq // 2), 1)

        self.block1 = nn.Sequential(
            nn.Conv2d(1, F1, (1, half_sfreq), padding=(0, half_sfreq // 2), bias=False),
            nn.BatchNorm2d(F1),
            nn.Conv2d(F1, F1 * D, (n_channels, 1), groups=F1, bias=False),
            nn.BatchNorm2d(F1 * D),
            nn.ELU(),
            nn.AvgPool2d((1, 4)),
            nn.Dropout(dropout),
        )

        self.block2 = nn.Sequential(
            nn.Conv2d(F1 * D, F1 * D, (1, 16), padding=(0, 8), groups=F1 * D, bias=False),
            nn.Conv2d(F1 * D, F2, (1, 1), bias=False),
            nn.BatchNorm2d(F2),
            nn.ELU(),
            nn.AvgPool2d((1, 8)),
            nn.Dropout(dropout),
        )

        with torch.no_grad():
            dummy = torch.zeros(1, 1, n_channels, n_samples)
            out = self.block2(self.block1(dummy))
            self.feature_dim = out.numel()

    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.block1(x)
        x = self.block2(x)
        return x.flatten(start_dim=1)


class SubjectAdapter(nn.Module):
    """A single per-subject affine transform (like a standalone BatchNorm1d without
    running stats) over the backbone's flattened features. Cheap to fine-tune alone."""

    def __init__(self, feature_dim):
        super().__init__()
        self.norm = nn.BatchNorm1d(feature_dim, affine=True)

    def forward(self, x):
        return self.norm(x)


class KaelvorNet(nn.Module):
    """
    use_ea is handled outside the model (as a preprocessing step on the numpy arrays);
    use_domain_head and use_adapter toggle those components for the ablation study.
    """

    def __init__(self, n_channels=22, n_classes=4, n_samples=1126, sfreq=250,
                 n_subjects=9, use_domain_head=True, use_adapter=True, grl_lambda=1.0):
        super().__init__()

        self.use_domain_head = use_domain_head
        self.use_adapter = use_adapter

        self.backbone = EEGNetBackbone(n_channels=n_channels, n_samples=n_samples, sfreq=sfreq)
        feat_dim = self.backbone.feature_dim

        if use_adapter:
            self.adapter = SubjectAdapter(feat_dim)

        self.classifier = nn.Linear(feat_dim, n_classes)

        if use_domain_head:
            self.grl = GradientReversalLayer(lambda_=grl_lambda)
            self.domain_head = nn.Sequential(
                nn.Linear(feat_dim, 32),
                nn.ELU(),
                nn.Linear(32, n_subjects),
            )

    def set_grl_lambda(self, lambda_):
        if self.use_domain_head:
            self.grl.lambda_ = lambda_

    def forward(self, x, return_domain_logits=False):

        features = self.backbone(x)

        adapted = self.adapter(features) if self.use_adapter else features

        class_logits = self.classifier(adapted)

        if return_domain_logits and self.use_domain_head:
            domain_logits = self.domain_head(self.grl(features))
            return class_logits, domain_logits

        return class_logits

---
## Training helper — supports the ablation toggles (EA / GRL / adapter)

In [8]:
# ==========================================================
# Training / Evaluation Helper
# ==========================================================

def normalize(X, mean, std):
    return (X - mean) / std


def train_kaelvornet(X_train, y_train, subj_train, X_test, y_test, n_subjects,
                      use_domain_head=True, use_adapter=True,
                      n_epochs=100, batch_size=32, lr=1e-3, grl_lambda_max=1.0):
    """
    X_train/X_test: (n_trials, n_channels, n_samples) -- already Euclidean-aligned upstream
    if EA is enabled for this run.
    subj_train: integer subject-id label per training trial (only used if use_domain_head).
    Returns (accuracy, kappa, model).
    """

    mean = X_train.mean(axis=(0, 2), keepdims=True)
    std = X_train.std(axis=(0, 2), keepdims=True) + 1e-8

    X_train_n = normalize(X_train, mean, std)
    X_test_n = normalize(X_test, mean, std)

    X_train_t = torch.tensor(X_train_n, dtype=torch.float32)
    y_train_t = torch.tensor(y_train, dtype=torch.long)
    subj_train_t = torch.tensor(subj_train, dtype=torch.long)
    X_test_t = torch.tensor(X_test_n, dtype=torch.float32).to(DEVICE)

    train_ds = TensorDataset(X_train_t, y_train_t, subj_train_t)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)

    model = KaelvorNet(
        n_channels=X_train.shape[1],
        n_classes=len(CLASS_NAMES),
        n_samples=X_train.shape[2],
        n_subjects=n_subjects,
        use_domain_head=use_domain_head,
        use_adapter=use_adapter,
    ).to(DEVICE)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    task_criterion = nn.CrossEntropyLoss()
    domain_criterion = nn.CrossEntropyLoss()

    model.train()

    for epoch in range(n_epochs):

        # Ramp lambda from 0 -> grl_lambda_max over training (stabilizes adversarial training)
        p = epoch / max(n_epochs - 1, 1)
        lambda_ = grl_lambda_max * (2.0 / (1.0 + np.exp(-10 * p)) - 1.0)
        model.set_grl_lambda(lambda_)

        epoch_loss = 0.0

        for xb, yb, sb in train_loader:

            xb, yb, sb = xb.to(DEVICE), yb.to(DEVICE), sb.to(DEVICE)

            optimizer.zero_grad()

            if use_domain_head:
                class_logits, domain_logits = model(xb, return_domain_logits=True)
                loss = task_criterion(class_logits, yb) + domain_criterion(domain_logits, sb)
            else:
                class_logits = model(xb)
                loss = task_criterion(class_logits, yb)

            loss.backward()
            optimizer.step()

            epoch_loss += loss.item() * xb.size(0)

        if (epoch + 1) % 25 == 0 or epoch == 0:
            print(f"    epoch {epoch+1}/{n_epochs}  loss={epoch_loss/len(train_ds):.4f}  "
                  f"(lambda={lambda_:.3f})")

    model.eval()

    with torch.no_grad():
        preds = model(X_test_t).argmax(dim=1).cpu().numpy()

    acc = accuracy_score(y_test, preds)
    kappa = cohen_kappa_score(y_test, preds)

    return acc, kappa, model, (mean, std)


def finetune_adapter(model, mean, std, X_few, y_few, n_epochs=50, lr=1e-3):
    """
    Few-shot cross-subject adaptation: freeze everything except the subject adapter
    (and classifier head), fine-tune on a handful of the new subject's labeled trials.
    """

    for name, param in model.named_parameters():
        param.requires_grad = ("adapter" in name) or ("classifier" in name)

    X_few_n = normalize(X_few, mean, std)
    X_few_t = torch.tensor(X_few_n, dtype=torch.float32).to(DEVICE)
    y_few_t = torch.tensor(y_few, dtype=torch.long).to(DEVICE)

    optimizer = torch.optim.Adam(
        [p for p in model.parameters() if p.requires_grad], lr=lr
    )
    criterion = nn.CrossEntropyLoss()

    model.train()

    for epoch in range(n_epochs):
        optimizer.zero_grad()
        logits = model(X_few_t)
        loss = criterion(logits, y_few_t)
        loss.backward()
        optimizer.step()

    model.eval()

    return model

---
## Evaluation 1 — Subject-Dependent (same protocol as the baselines)

Full KaelvorNet (EA + GRL + adapter), pretrained *including* the target subject's own T-file
data, tested on that subject's E-file (or the 80/20 fallback).

In [9]:
# ==========================================================
# Subject-Dependent Evaluation
# ==========================================================

subject_ids_all = {sid: i for i, sid in enumerate(subjects_covered)}
n_subjects_total = len(subject_ids_all)

subject_dependent_results = []

for t_file in mi_files:

    subject_id = t_file.stem[:-1]
    subj_int = subject_ids_all[subject_id]

    e_file = next((f for f in eval_files if f.stem[:-1] == subject_id), None)
    if e_file is None:
        print(f"[{subject_id}] Skipped -- no matching *E file found.")
        continue

    X_train, y_train = extract_epochs(t_file)
    true_labels = find_true_labels(subject_id, mat_search_root)

    used_fallback = true_labels is None

    if used_fallback:
        X_tr, X_te, y_tr, y_te = train_test_split(
            X_train, y_train, test_size=0.2, stratify=y_train, random_state=SEED
        )
        X_train, y_train, X_test, y_test = X_tr, y_tr, X_te, y_te
    else:
        X_test = extract_epochs_eval(e_file)
        if len(true_labels) != len(X_test):
            print(f"[{subject_id}] Skipped -- label/epoch count mismatch.")
            continue
        y_test = true_labels

    # Euclidean Alignment -- fit on this subject's training trials, apply to both
    X_train_a, X_test_a = euclidean_align(X_train, X_test)

    subj_train = np.full(len(y_train), subj_int)

    print(f"[{subject_id}] Training KaelvorNet ({'fallback split' if used_fallback else 'official split'})...")

    acc, kappa, _, _ = train_kaelvornet(
        X_train_a, y_train, subj_train, X_test_a, y_test,
        n_subjects=n_subjects_total, use_domain_head=True, use_adapter=True
    )

    subject_dependent_results.append({
        "Subject": subject_id, "Accuracy": acc, "Kappa": kappa,
        "Split": "Fallback (80/20 within T)" if used_fallback else "Official (T -> E)"
    })

    print(f"[{subject_id}] KaelvorNet acc={acc:.3f}  kappa={kappa:.3f}\n")

subject_dependent_df = pd.DataFrame(subject_dependent_results)
display(subject_dependent_df)

if len(subject_dependent_df):
    print(f"Mean Accuracy: {subject_dependent_df['Accuracy'].mean():.3f}  "
          f"Mean Kappa: {subject_dependent_df['Kappa'].mean():.3f}")

/tmp/ipykernel_58/600538519.py:8: RuntimeWarning: This filename (/kaggle/input/notebooks/sababahoquesaba/eeg-preprocessing-pipeline/preprocessed_dataset/A01/A01T.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  raw = mne.io.read_raw_fif(fif_file, preload=True, verbose=False)


[A01] Training KaelvorNet (fallback split)...
    epoch 1/100  loss=3.2029  (lambda=0.000)
    epoch 25/100  loss=1.6426  (lambda=0.837)
    epoch 50/100  loss=0.9203  (lambda=0.986)
    epoch 75/100  loss=0.5184  (lambda=0.999)
    epoch 100/100  loss=0.3168  (lambda=1.000)
[A01] KaelvorNet acc=0.724  kappa=0.632



/tmp/ipykernel_58/600538519.py:8: RuntimeWarning: This filename (/kaggle/input/notebooks/sababahoquesaba/eeg-preprocessing-pipeline/preprocessed_dataset/A02/A02T.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  raw = mne.io.read_raw_fif(fif_file, preload=True, verbose=False)


[A02] Training KaelvorNet (fallback split)...
    epoch 1/100  loss=3.3007  (lambda=0.000)
    epoch 25/100  loss=1.2581  (lambda=0.837)
    epoch 50/100  loss=0.7426  (lambda=0.986)
    epoch 75/100  loss=0.4676  (lambda=0.999)
    epoch 100/100  loss=0.2498  (lambda=1.000)
[A02] KaelvorNet acc=0.552  kappa=0.402



/tmp/ipykernel_58/600538519.py:8: RuntimeWarning: This filename (/kaggle/input/notebooks/sababahoquesaba/eeg-preprocessing-pipeline/preprocessed_dataset/A03/A03T.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  raw = mne.io.read_raw_fif(fif_file, preload=True, verbose=False)


...

[A08] Training KaelvorNet (fallback split)...
    epoch 1/100  loss=3.1712  (lambda=0.000)
    epoch 25/100  loss=1.6247  (lambda=0.837)
    epoch 50/100  loss=0.5717  (lambda=0.986)
    epoch 75/100  loss=0.3392  (lambda=0.999)
    epoch 100/100  loss=0.1916  (lambda=1.000)
[A08] KaelvorNet acc=0.776  kappa=0.701



/tmp/ipykernel_58/600538519.py:8: RuntimeWarning: This filename (/kaggle/input/notebooks/sababahoquesaba/eeg-preprocessing-pipeline/preprocessed_dataset/A09/A09T.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  raw = mne.io.read_raw_fif(fif_file, preload=True, verbose=False)


[A09] Training KaelvorNet (fallback split)...
    epoch 1/100  loss=3.3821  (lambda=0.000)
    epoch 25/100  loss=1.0982  (lambda=0.837)
    epoch 50/100  loss=0.7978  (lambda=0.986)
    epoch 75/100  loss=0.5774  (lambda=0.999)
    epoch 100/100  loss=0.2857  (lambda=1.000)
[A09] KaelvorNet acc=0.724  kappa=0.631



,Subject,Accuracy,Kappa,Split
0,A01,0.724138,0.631892,Fallback (80/20 within T)
1,A02,0.551724,0.401825,Fallback (80/20 within T)
2,A03,0.637931,0.518196,Fallback (80/20 within T)
3,A04,0.655172,0.539134,Fallback (80/20 within T)
4,A05,0.827586,0.770388,Fallback (80/20 within T)
5,A06,0.448276,0.261734,Fallback (80/20 within T)
6,A07,0.793103,0.723919,Fallback (80/20 within T)
7,A08,0.775862,0.701149,Fallback (80/20 within T)
8,A09,0.724138,0.631454,Fallback (80/20 within T)


Mean Accuracy: 0.682  Mean Kappa: 0.576


---
## Evaluation 2 — Cross-Subject Leave-One-Subject-Out

Pretrain on 8 subjects' T-files (holding one out entirely), then test on the held-out
subject's E-file two ways: **zero-shot** (frozen model, no fine-tuning) and **few-shot**
(fine-tune only the subject adapter on a handful of labeled trials from that subject).

In [10]:
# ==========================================================
# Cross-Subject (Leave-One-Subject-Out) Evaluation
# ==========================================================

FEW_SHOT_TRIALS_PER_CLASS = 15  # tune this -- see "Adapter capacity trade-off" in the design doc

loso_results = []

for held_out_id in subjects_covered:

    held_out_int = subject_ids_all[held_out_id]

    train_subject_ids = [s for s in subjects_covered if s != held_out_id]

    # ---- Build the pretraining pool from the other 8 subjects ----
    X_pool, y_pool, subj_pool = [], [], []

    for sid in train_subject_ids:

        t_file = next(f for f in mi_files if f.stem[:-1] == sid)
        X_s, y_s = extract_epochs(t_file)

        # Align each pretraining subject to their own reference (EA is per-subject)
        X_s_aligned, = euclidean_align(X_s)

        X_pool.append(X_s_aligned)
        y_pool.append(y_s)
        subj_pool.append(np.full(len(y_s), subject_ids_all[sid]))

    X_pool = np.concatenate(X_pool, axis=0)
    y_pool = np.concatenate(y_pool, axis=0)
    subj_pool = np.concatenate(subj_pool, axis=0)

    # ---- Held-out subject's data ----
    t_file_ho = next(f for f in mi_files if f.stem[:-1] == held_out_id)
    e_file_ho = next((f for f in eval_files if f.stem[:-1] == held_out_id), None)

    if e_file_ho is None:
        print(f"[{held_out_id}] Skipped -- no matching *E file found.")
        continue

    X_ho_train, y_ho_train = extract_epochs(t_file_ho)  # used only for few-shot / fallback

    true_labels = find_true_labels(held_out_id, mat_search_root)
    used_fallback = true_labels is None

    if used_fallback:
        X_ho_few, X_ho_test, y_ho_few, y_ho_test = train_test_split(
            X_ho_train, y_ho_train, test_size=0.7, stratify=y_ho_train, random_state=SEED
        )
    else:
        X_ho_test = extract_epochs_eval(e_file_ho)
        if len(true_labels) != len(X_ho_test):
            print(f"[{held_out_id}] Skipped -- label/epoch count mismatch.")
            continue
        y_ho_test = true_labels
        X_ho_few, y_ho_few = X_ho_train, y_ho_train  # full T file available for few-shot sampling

    # Align held-out subject's own data to its own reference
    X_ho_test_a, X_ho_few_a = euclidean_align(X_ho_test, X_ho_few)

    print(f"[{held_out_id}] Pretraining on {len(train_subject_ids)} subjects "
          f"({'fallback' if used_fallback else 'official'} test split for held-out subject)...")

    zero_shot_acc, zero_shot_kappa, model, (mean, std) = train_kaelvornet(
        X_pool, y_pool, subj_pool, X_ho_test_a, y_ho_test,
        n_subjects=n_subjects_total, use_domain_head=True, use_adapter=True
    )

    print(f"[{held_out_id}] Zero-shot   acc={zero_shot_acc:.3f}  kappa={zero_shot_kappa:.3f}")

    # ---- Few-shot adapter fine-tuning ----
    few_X, few_y = [], []
    for c in range(len(CLASS_NAMES)):
        idx = np.where(y_ho_few == c)[0][:FEW_SHOT_TRIALS_PER_CLASS]
        few_X.append(X_ho_few_a[idx])
        few_y.append(y_ho_few[idx])

    X_few = np.concatenate(few_X, axis=0)
    y_few = np.concatenate(few_y, axis=0)

    model = finetune_adapter(model, mean, std, X_few, y_few)

    X_ho_test_n = normalize(X_ho_test_a, mean, std)
    X_ho_test_t = torch.tensor(X_ho_test_n, dtype=torch.float32).to(DEVICE)

    with torch.no_grad():
        few_shot_preds = model(X_ho_test_t).argmax(dim=1).cpu().numpy()

    few_shot_acc = accuracy_score(y_ho_test, few_shot_preds)
    few_shot_kappa = cohen_kappa_score(y_ho_test, few_shot_preds)

    print(f"[{held_out_id}] Few-shot    acc={few_shot_acc:.3f}  kappa={few_shot_kappa:.3f}\n")

    loso_results.append({
        "Held_Out_Subject": held_out_id,
        "Zero_Shot_Accuracy": zero_shot_acc,
        "Zero_Shot_Kappa": zero_shot_kappa,
        "Few_Shot_Accuracy": few_shot_acc,
        "Few_Shot_Kappa": few_shot_kappa,
        "Split": "Fallback" if used_fallback else "Official",
    })

loso_df = pd.DataFrame(loso_results)
display(loso_df)

if len(loso_df):
    print(f"\nMean Zero-Shot Accuracy: {loso_df['Zero_Shot_Accuracy'].mean():.3f}")
    print(f"Mean Few-Shot Accuracy : {loso_df['Few_Shot_Accuracy'].mean():.3f}")

/tmp/ipykernel_58/600538519.py:8: RuntimeWarning: This filename (/kaggle/input/notebooks/sababahoquesaba/eeg-preprocessing-pipeline/preprocessed_dataset/A02/A02T.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  raw = mne.io.read_raw_fif(fif_file, preload=True, verbose=False)
/tmp/ipykernel_58/600538519.py:8: RuntimeWarning: This filename (/kaggle/input/notebooks/sababahoquesaba/eeg-preprocessing-pipeline/preprocessed_dataset/A03/A03T.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  raw = mne.io.read_raw_fif(fif_file, preload=True, verbose=False)
/tmp/ipykernel_58/600538519.py:8: RuntimeWarning: This filename (/kaggle/input

[A01] Pretraining on 8 subjects (fallback test split for held-out subject)...
    epoch 1/100  loss=3.2566  (lambda=0.000)
    epoch 25/100  loss=2.7740  (lambda=0.837)
    epoch 50/100  loss=2.8416  (lambda=0.986)
    epoch 75/100  loss=2.8562  (lambda=0.999)
    epoch 100/100  loss=2.8054  (lambda=1.000)
[A01] Zero-shot   acc=0.579  kappa=0.439
[A01] Few-shot    acc=0.629  kappa=0.505



/tmp/ipykernel_58/600538519.py:8: RuntimeWarning: This filename (/kaggle/input/notebooks/sababahoquesaba/eeg-preprocessing-pipeline/preprocessed_dataset/A01/A01T.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  raw = mne.io.read_raw_fif(fif_file, preload=True, verbose=False)
/tmp/ipykernel_58/600538519.py:8: RuntimeWarning: This filename (/kaggle/input/notebooks/sababahoquesaba/eeg-preprocessing-pipeline/preprocessed_dataset/A03/A03T.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  raw = mne.io.read_raw_fif(fif_file, preload=True, verbose=False)
/tmp/ipykernel_58/600538519.py:8: RuntimeWarning: This filename (/kaggle/input

[A02] Pretraining on 8 subjects (fallback test split for held-out subject)...
    epoch 1/100  loss=3.2824  (lambda=0.000)
    epoch 25/100  loss=2.9129  (lambda=0.837)
    epoch 50/100  loss=2.9000  (lambda=0.986)
    epoch 75/100  loss=2.8741  (lambda=0.999)
    epoch 100/100  loss=2.8518  (lambda=1.000)
[A02] Zero-shot   acc=0.673  kappa=0.564
[A02] Few-shot    acc=0.649  kappa=0.531



/tmp/ipykernel_58/600538519.py:8: RuntimeWarning: This filename (/kaggle/input/notebooks/sababahoquesaba/eeg-preprocessing-pipeline/preprocessed_dataset/A01/A01T.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  raw = mne.io.read_raw_fif(fif_file, preload=True, verbose=False)
/tmp/ipykernel_58/600538519.py:8: RuntimeWarning: This filename (/kaggle/input/notebooks/sababahoquesaba/eeg-preprocessing-pipeline/preprocessed_dataset/A02/A02T.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  raw = mne.io.read_raw_fif(fif_file, preload=True, verbose=False)
/tmp/ipykernel_58/600538519.py:8: RuntimeWarning: This filename (/kaggle/input

...

[A08] Pretraining on 8 subjects (fallback test split for held-out subject)...
    epoch 1/100  loss=3.2339  (lambda=0.000)
    epoch 25/100  loss=2.9481  (lambda=0.837)
    epoch 50/100  loss=2.9057  (lambda=0.986)
    epoch 75/100  loss=2.8513  (lambda=0.999)
    epoch 100/100  loss=2.8358  (lambda=1.000)
[A08] Zero-shot   acc=0.599  kappa=0.466
[A08] Few-shot    acc=0.653  kappa=0.538



/tmp/ipykernel_58/600538519.py:8: RuntimeWarning: This filename (/kaggle/input/notebooks/sababahoquesaba/eeg-preprocessing-pipeline/preprocessed_dataset/A01/A01T.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  raw = mne.io.read_raw_fif(fif_file, preload=True, verbose=False)
/tmp/ipykernel_58/600538519.py:8: RuntimeWarning: This filename (/kaggle/input/notebooks/sababahoquesaba/eeg-preprocessing-pipeline/preprocessed_dataset/A02/A02T.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  raw = mne.io.read_raw_fif(fif_file, preload=True, verbose=False)
/tmp/ipykernel_58/600538519.py:8: RuntimeWarning: This filename (/kaggle/input

[A09] Pretraining on 8 subjects (fallback test split for held-out subject)...
    epoch 1/100  loss=3.2566  (lambda=0.000)
    epoch 25/100  loss=2.8150  (lambda=0.837)
    epoch 50/100  loss=2.8218  (lambda=0.986)
    epoch 75/100  loss=2.8388  (lambda=0.999)
    epoch 100/100  loss=2.8386  (lambda=1.000)
[A09] Zero-shot   acc=0.545  kappa=0.393
[A09] Few-shot    acc=0.589  kappa=0.452



,Held_Out_Subject,Zero_Shot_Accuracy,Zero_Shot_Kappa,Few_Shot_Accuracy,Few_Shot_Kappa,Split
0,A01,0.579208,0.438650,0.628713,0.504902,Fallback
1,A02,0.673267,0.564285,0.648515,0.531261,Fallback
2,A03,0.514851,0.352458,0.500000,0.333529,Fallback
3,A04,0.589109,0.452038,0.599010,0.465277,Fallback
4,A05,0.643564,0.524317,0.762376,0.683106,Fallback
5,A06,0.608911,0.478939,0.599010,0.465364,Fallback
6,A07,0.707921,0.610561,0.757426,0.676462,Fallback
7,A08,0.599010,0.466114,0.653465,0.538014,Fallback
8,A09,0.544554,0.392521,0.589109,0.452181,Fallback



Mean Zero-Shot Accuracy: 0.607
Mean Few-Shot Accuracy : 0.638


---
## Ablation Study

Reproduces the ablation table from the design doc by toggling EA / GRL / adapter, evaluated
on the leave-one-subject-out zero-shot setting (the clearest test of cross-subject
generalization) for one representative held-out subject at a time. Uses a subset of subjects
by default to keep runtime reasonable -- widen `ABLATION_SUBJECTS` to run the full sweep.

In [11]:
# ==========================================================
# Ablation Study
# ==========================================================

ABLATION_SUBJECTS = subjects_covered[:3]  # widen this list for a full sweep

ABLATION_VARIANTS = [
    {"name": "EEGNet baseline",        "use_ea": False, "use_domain_head": False, "use_adapter": False},
    {"name": "+ EA only",              "use_ea": True,  "use_domain_head": False, "use_adapter": False},
    {"name": "+ EA + GRL",             "use_ea": True,  "use_domain_head": True,  "use_adapter": False},
    {"name": "+ EA + GRL + adapter",   "use_ea": True,  "use_domain_head": True,  "use_adapter": True},
    {"name": "+ GRL + adapter, no EA", "use_ea": False, "use_domain_head": True,  "use_adapter": True},
]

ablation_results = []

for held_out_id in ABLATION_SUBJECTS:

    train_subject_ids = [s for s in subjects_covered if s != held_out_id]

    for variant in ABLATION_VARIANTS:

        X_pool, y_pool, subj_pool = [], [], []

        for sid in train_subject_ids:

            t_file = next(f for f in mi_files if f.stem[:-1] == sid)
            X_s, y_s = extract_epochs(t_file)

            if variant["use_ea"]:
                X_s, = euclidean_align(X_s)

            X_pool.append(X_s)
            y_pool.append(y_s)
            subj_pool.append(np.full(len(y_s), subject_ids_all[sid]))

        X_pool = np.concatenate(X_pool, axis=0)
        y_pool = np.concatenate(y_pool, axis=0)
        subj_pool = np.concatenate(subj_pool, axis=0)

        e_file_ho = next((f for f in eval_files if f.stem[:-1] == held_out_id), None)
        if e_file_ho is None:
            continue

        true_labels = find_true_labels(held_out_id, mat_search_root)

        if true_labels is None:
            t_file_ho = next(f for f in mi_files if f.stem[:-1] == held_out_id)
            X_ho, y_ho = extract_epochs(t_file_ho)
            _, X_ho_test, _, y_ho_test = train_test_split(
                X_ho, y_ho, test_size=0.7, stratify=y_ho, random_state=SEED
            )
        else:
            X_ho_test = extract_epochs_eval(e_file_ho)
            y_ho_test = true_labels

        if variant["use_ea"]:
            X_ho_test, = euclidean_align(X_ho_test)

        print(f"[{held_out_id}] {variant['name']} ...")

        acc, kappa, _, _ = train_kaelvornet(
            X_pool, y_pool, subj_pool, X_ho_test, y_ho_test,
            n_subjects=n_subjects_total,
            use_domain_head=variant["use_domain_head"],
            use_adapter=variant["use_adapter"],
            n_epochs=60,  # shorter run for the ablation sweep
        )

        ablation_results.append({
            "Held_Out_Subject": held_out_id,
            "Variant": variant["name"],
            "Accuracy": acc,
            "Kappa": kappa,
        })

        print(f"    acc={acc:.3f}  kappa={kappa:.3f}")

ablation_df = pd.DataFrame(ablation_results)
display(ablation_df.pivot_table(index="Variant", values=["Accuracy", "Kappa"], aggfunc="mean"))

/tmp/ipykernel_58/600538519.py:8: RuntimeWarning: This filename (/kaggle/input/notebooks/sababahoquesaba/eeg-preprocessing-pipeline/preprocessed_dataset/A02/A02T.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  raw = mne.io.read_raw_fif(fif_file, preload=True, verbose=False)
/tmp/ipykernel_58/600538519.py:8: RuntimeWarning: This filename (/kaggle/input/notebooks/sababahoquesaba/eeg-preprocessing-pipeline/preprocessed_dataset/A03/A03T.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  raw = mne.io.read_raw_fif(fif_file, preload=True, verbose=False)
/tmp/ipykernel_58/600538519.py:8: RuntimeWarning: This filename (/kaggle/input

[A01] EEGNet baseline ...
    epoch 1/60  loss=1.3376  (lambda=0.000)
    epoch 25/60  loss=0.6237  (lambda=0.966)
    epoch 50/60  loss=0.5807  (lambda=1.000)
    acc=0.673  kappa=0.564


/tmp/ipykernel_58/600538519.py:8: RuntimeWarning: This filename (/kaggle/input/notebooks/sababahoquesaba/eeg-preprocessing-pipeline/preprocessed_dataset/A02/A02T.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  raw = mne.io.read_raw_fif(fif_file, preload=True, verbose=False)
/tmp/ipykernel_58/600538519.py:8: RuntimeWarning: This filename (/kaggle/input/notebooks/sababahoquesaba/eeg-preprocessing-pipeline/preprocessed_dataset/A03/A03T.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  raw = mne.io.read_raw_fif(fif_file, preload=True, verbose=False)
/tmp/ipykernel_58/600538519.py:8: RuntimeWarning: This filename (/kaggle/input

[A01] + EA only ...
    epoch 1/60  loss=1.3837  (lambda=0.000)
    epoch 25/60  loss=0.6925  (lambda=0.966)
    epoch 50/60  loss=0.5864  (lambda=1.000)
    acc=0.624  kappa=0.499


/tmp/ipykernel_58/600538519.py:8: RuntimeWarning: This filename (/kaggle/input/notebooks/sababahoquesaba/eeg-preprocessing-pipeline/preprocessed_dataset/A02/A02T.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  raw = mne.io.read_raw_fif(fif_file, preload=True, verbose=False)
/tmp/ipykernel_58/600538519.py:8: RuntimeWarning: This filename (/kaggle/input/notebooks/sababahoquesaba/eeg-preprocessing-pipeline/preprocessed_dataset/A03/A03T.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  raw = mne.io.read_raw_fif(fif_file, preload=True, verbose=False)
/tmp/ipykernel_58/600538519.py:8: RuntimeWarning: This filename (/kaggle/input

...

/tmp/ipykernel_58/600538519.py:8: RuntimeWarning: This filename (/kaggle/input/notebooks/sababahoquesaba/eeg-preprocessing-pipeline/preprocessed_dataset/A01/A01T.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  raw = mne.io.read_raw_fif(fif_file, preload=True, verbose=False)
/tmp/ipykernel_58/600538519.py:8: RuntimeWarning: This filename (/kaggle/input/notebooks/sababahoquesaba/eeg-preprocessing-pipeline/preprocessed_dataset/A02/A02T.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  raw = mne.io.read_raw_fif(fif_file, preload=True, verbose=False)
/tmp/ipykernel_58/600538519.py:8: RuntimeWarning: This filename (/kaggle/input

[A03] + EA + GRL + adapter ...
    epoch 1/60  loss=3.2750  (lambda=0.000)
    epoch 25/60  loss=2.9284  (lambda=0.966)
    epoch 50/60  loss=2.8030  (lambda=1.000)
    acc=0.470  kappa=0.293


/tmp/ipykernel_58/600538519.py:8: RuntimeWarning: This filename (/kaggle/input/notebooks/sababahoquesaba/eeg-preprocessing-pipeline/preprocessed_dataset/A01/A01T.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  raw = mne.io.read_raw_fif(fif_file, preload=True, verbose=False)
/tmp/ipykernel_58/600538519.py:8: RuntimeWarning: This filename (/kaggle/input/notebooks/sababahoquesaba/eeg-preprocessing-pipeline/preprocessed_dataset/A02/A02T.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  raw = mne.io.read_raw_fif(fif_file, preload=True, verbose=False)
/tmp/ipykernel_58/600538519.py:8: RuntimeWarning: This filename (/kaggle/input

[A03] + GRL + adapter, no EA ...
    epoch 1/60  loss=3.2952  (lambda=0.000)
    epoch 25/60  loss=2.7830  (lambda=0.966)
    epoch 50/60  loss=2.8284  (lambda=1.000)
    acc=0.554  kappa=0.406


,Accuracy,Kappa
Variant,,
+ EA + GRL,0.500000,0.332946
+ EA + GRL + adapter,0.524752,0.366089
+ EA only,0.636964,0.516044
"+ GRL + adapter, no EA",0.577558,0.436659
EEGNet baseline,0.666667,0.555607


---
## Save Results

In [12]:
# ==========================================================
# Save Results
# ==========================================================

subject_dependent_df.to_csv(OUTPUT_PATH / "kaelvornet_subject_dependent_results.csv", index=False)
loso_df.to_csv(OUTPUT_PATH / "kaelvornet_loso_results.csv", index=False)
ablation_df.to_csv(OUTPUT_PATH / "kaelvornet_ablation_results.csv", index=False)

print("Saved:")
print(" ", OUTPUT_PATH / "kaelvornet_subject_dependent_results.csv")
print(" ", OUTPUT_PATH / "kaelvornet_loso_results.csv")
print(" ", OUTPUT_PATH / "kaelvornet_ablation_results.csv")

Saved:
  /kaggle/working/kaelvornet_subject_dependent_results.csv
  /kaggle/working/kaelvornet_loso_results.csv
  /kaggle/working/kaelvornet_ablation_results.csv
